# Weighted competing pricing from an arbitrary Parquet file

This notebook applies the competing-pricing procedure directly to one Parquet file. The file must contain one row per policy or risk, with two predictions, exposure, and observed loss. Column names are configurable below.

Each prediction is first aligned to the observed losses so that its global ELR is exactly one before commercial loadings. Choice probabilities are then computed from the loaded quotes and used as portfolio weights.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

## Input and pricing settings

Set `PARQUET_PATH` and the four column names to match the input file. An identifier column is optional and is only used for displaying the input preview.

In [ ]:
PARQUET_PATH = Path("data/03_predictions/competing_pricing.parquet")
ID_COLUMN = None
PREDICTION_A_COLUMN = "prediction_a"
PREDICTION_B_COLUMN = "prediction_b"
EXPOSURE_COLUMN = "exposure"
LOSS_COLUMN = "loss"

LOADING_A = 1.0
LOADING_B = 1.0
ELASTICITY = 2.0
ALIGN_PREDICTIONS = True

if not PARQUET_PATH.is_file():
    raise FileNotFoundError(f"Parquet file not found: {PARQUET_PATH}")
if ELASTICITY <= 0:
    raise ValueError("ELASTICITY must be positive.")
if LOADING_A <= 0 or LOADING_B <= 0:
    raise ValueError("Commercial loadings must be positive.")

In [ ]:
required_columns = [
    PREDICTION_A_COLUMN,
    PREDICTION_B_COLUMN,
    EXPOSURE_COLUMN,
    LOSS_COLUMN,
]
data = pd.read_parquet(PARQUET_PATH)
missing_columns = sorted(set(required_columns) - set(data.columns))
if missing_columns:
    raise ValueError(f"Missing required columns: {missing_columns}")

portfolio = data[required_columns].copy()
portfolio.columns = ["prediction_a", "prediction_b", "exposure", "loss"]
if portfolio.empty:
    raise ValueError("The input Parquet file contains no rows.")
if portfolio.isna().any().any():
    raise ValueError("Required input columns must not contain missing values.")
if not np.isfinite(portfolio.to_numpy(dtype=float)).all():
    raise ValueError("Required input columns must contain finite numeric values.")
if (portfolio[["prediction_a", "prediction_b", "exposure"]] <= 0).any().any():
    raise ValueError("Predictions and exposure must be positive.")
if (portfolio["loss"] < 0).any():
    raise ValueError("Loss must be non-negative.")

print(f"Rows loaded: {len(portfolio):,}")
portfolio.head()

## Align global ELR

For each competitor $j$, the alignment factor is:

$$c_j = \frac{\sum_i L_i}{\sum_i \widehat{R}_{ij}E_i}.$$

After multiplying the prediction by $c_j$, its global loss ratio before commercial loading is exactly one.

In [ ]:
def align_prediction(prediction, exposure, loss):
    factor = loss.sum() / (prediction * exposure).sum()
    return prediction * factor, factor


if ALIGN_PREDICTIONS:
    portfolio["prediction_a"], factor_a = align_prediction(
        portfolio["prediction_a"], portfolio["exposure"], portfolio["loss"]
    )
    portfolio["prediction_b"], factor_b = align_prediction(
        portfolio["prediction_b"], portfolio["exposure"], portfolio["loss"]
    )
else:
    factor_a = factor_b = 1.0

portfolio["quote_a"] = LOADING_A * portfolio["prediction_a"]
portfolio["quote_b"] = LOADING_B * portfolio["prediction_b"]
portfolio["weight_a"] = 1 / (1 + (portfolio["quote_a"] / portfolio["quote_b"]) ** ELASTICITY)
portfolio["weight_b"] = 1 - portfolio["weight_a"]

print(f"ELR alignment factors: A={factor_a:.6g}, B={factor_b:.6g}")
portfolio[["prediction_a", "prediction_b", "quote_a", "quote_b", "weight_a", "weight_b"]].head()

## Probability-weighted KPIs

For each competitor, won-business totals use its choice probability as a weight. Lost-business totals use the complementary weight.

In [ ]:
def safe_ratio(numerator, denominator):
    return numerator / denominator if denominator > 0 else np.nan


def weighted_kpis(portfolio, competitor):
    weight = portfolio[f"weight_{competitor}"]
    complement = 1 - weight
    premium = portfolio[f"quote_{competitor}"] * portfolio["exposure"]
    won_loss = (weight * portfolio["loss"]).sum()
    won_premium = (weight * premium).sum()
    lost_loss = (complement * portfolio["loss"]).sum()
    lost_premium = (complement * premium).sum()
    profit = won_premium - won_loss
    return {
        "competitor": competitor.upper(),
        "expected_n_won": weight.sum(),
        "expected_share_won": weight.mean(),
        "lr_win": safe_ratio(won_loss, won_premium),
        "lr_lose": safe_ratio(lost_loss, lost_premium),
        "profit": profit,
        "profit_rate": safe_ratio(profit, portfolio["exposure"].sum()),
        "expected_won_premium": won_premium,
        "expected_won_loss": won_loss,
    }

weighted_results = pd.DataFrame([
    weighted_kpis(portfolio, "a"),
    weighted_kpis(portfolio, "b"),
]).set_index("competitor")
weighted_results.round(4)

In [ ]:
weighted_results[["lr_win", "lr_lose"]].plot.bar(
    figsize=(9, 5),
    title="Probability-weighted loss ratios",
    ylabel="Loss ratio",
    rot=0,
)
plt.grid(axis="y", alpha=0.25)
plt.show()

weighted_results[["profit_rate"]].plot.bar(
    figsize=(7, 4),
    title="Probability-weighted profit rate",
    ylabel="Profit / offered exposure",
    rot=0,
)
plt.axhline(0, color="black", linewidth=0.8)
plt.grid(axis="y", alpha=0.25)
plt.show()